# Kiểm tra Qwen2-Audio INT8 (ref_runtime) — sổ tay đã sắp theo chức năng
Mỗi nhóm có một **mã chữ cái**; mỗi ô trong nhóm có mã `[Xn]` ở dòng đầu. Chạy theo thứ tự A → (B) → F; C, D, E là các kiểm tra đã làm xong (kết quả đã ghi vào Project), chỉ chạy lại khi cần.

| Mã | Chức năng | Trạng thái | File cần upload (ngoài `ref_runtime.py`, `make_cpp_inputs.py`) |
|---|---|---|---|
| **A · SETUP** | Drive, thư viện, đường dẫn gói, kiểm tra file | luôn chạy đầu phiên | |
| **B · INPUT** | Tạo `in_clip00` (template prompt + hàng audio) cho board | chỉ khi thiếu `in_clip00` | |
| **C · ENC** | Encoder: ref float32/float64 so với board trên `01.wav` | đã xong | `01.wav`, `rows_board*.f32`, `mel_board.f32` (từ board) |
| **D · TXT** | Chữ sinh của ref (INT8 mô phỏng) cho vài clip | đã xong | các `NN.wav` cần chạy |
| **E · SCORE** | Chấm điểm cả bộ 14 clip trên board so với câu gốc FP32 | đã xong | `score_wav_set.py`, `answers.txt`, `results.tsv` |
| **F · SWAP** | Hoán đổi FP32 ↔ INT8 giữa encoder và LLM để biết lỗi nằm ở đâu (E1/E2/E3) | **đang làm** | `01.wav`, `rows_board.f32` |

## A · SETUP

In [ ]:
# [A1] Kết nối Drive, cài thư viện, đường dẫn gói
from google.colab import drive; drive.mount('/content/drive')
!pip -q install scipy transformers
PKG = "<PKG_DIR>"

Mounted at /content/drive


In [ ]:
# [A2] Upload script (chạy lại sau mỗi lần Colab reset)
from google.colab import files; files.upload()      # chọn ref_runtime.py, make_cpp_inputs.py

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
# [A3] Kiểm tra những gì đang có
import os
for f in ["ref_runtime.py", "make_cpp_inputs.py"]: print(f"{f:22s}", "OK" if os.path.exists(f) else "THIẾU -> chạy A2")
print(f"{'PKG/MANIFEST.json':22s}", "OK" if os.path.exists(PKG + "/MANIFEST.json") else "KHÔNG THẤY -> kiểm tra PKG ở A1 / mount Drive")
print(f"{'in_clip00/ids.i32':22s}", "OK" if os.path.exists("in_clip00/ids.i32") else "thiếu -> chạy B1")
print(f"{'01.wav':22s}", "OK" if os.path.exists("01.wav") else "thiếu -> upload 01.wav trước khi chạy C/F")

ref_runtime.py         OK
make_cpp_inputs.py     OK
PKG/MANIFEST.json      OK
in_clip00/ids.i32      OK
01.wav                 OK


## B · INPUT — gói đầu vào cho board
`in_clip00/ids.i32` là template prompt dùng cho **mọi** clip trên board (`--template in_clip00/ids.i32`). Chỉ chạy khi `in_clip00` bị mất.
Các gói clip 12/24/37/49 đã tạo, đã chạy trên board và đã ghi kết quả — đã bỏ khỏi sổ tay.

In [ ]:
# [B1] Tạo in_clip00 nếu chưa có (~1 phút)
import os
if os.path.exists("in_clip00/ids.i32"): print("in_clip00 đã có, bỏ qua")
else:
    !python make_cpp_inputs.py make --pkg {PKG} --clip 0 --out in_clip00 --ref-logits

   encoder: mel 688 -> 344 -> 172 hang | 42.6s
ids 204 | rows (172, 896) | gen_fp32 91 | gen_int8 92 -> in_clip00
ref top5 -> in_clip00/ref_top5.txt


## C · ENC — encoder: ref so với board trên `01.wav`  (đã xong, có thể bỏ qua)
Kết luận đã ghi trong Project: board nằm ở mức nhiễu vốn có của encoder (rms tương đối 0.140 so với 0.132 giữa ref float32 và float64); mel khớp 6e-8; board chạy xác định.
Cần: `01.wav` và các file dump từ board (`rows_board.f32`, `rows_board2.f32`, `rows_board_exact.f32`, `mel_board.f32`).

In [ ]:
# [C1] Encoder ref float32 và float64 trên 01.wav (~1 phút mỗi lệnh)
!python3 ref_runtime.py run --pkg {PKG} --wav 01.wav --rows-only --dump-rows rows_ref.f32 --dump-mel mel_ref.f32
!python3 ref_runtime.py run --pkg {PKG} --wav 01.wav --rows-only --hp --dump-rows rows_ref_hp.f32

In [ ]:
# [C2] So sánh rows/mel (dùng lại hàm cmp ở các nhóm sau)
import numpy as np, os
H = 896
ld = lambda f: np.fromfile(f, np.float32).reshape(-1, H)
def cmp(a, b, n):
    d = a - b
    print(f"{n:36s} rms tương đối {np.sqrt((d**2).mean()/(b**2).mean()):.2e} | max|diff| {abs(d).max():.3f} | cos hàng min {min((x@y)/np.linalg.norm(x)/np.linalg.norm(y) for x, y in zip(a, b)):.4f}")
if os.path.exists("mel_board.f32"):
    mb = np.fromfile("mel_board.f32", np.float32).reshape(128, -1); mr = np.fromfile("mel_ref.f32", np.float32).reshape(128, -1); n = min(mb.shape[1], mr.shape[1])
    print("mel board vs ref: max|diff|", abs(mb[:, :n] - mr[:, :n]).max(), "| cột", mb.shape[1], mr.shape[1])
R_, RH = ld("rows_ref.f32"), ld("rows_ref_hp.f32"); cmp(R_, RH, "ref f32 vs ref float64 (nhiễu ulp)")
for nm, f in [("board", "rows_board.f32"), ("board (GELU erff)", "rows_board_exact.f32")]:
    if os.path.exists(f): cmp(ld(f), R_, f"{nm} vs ref")
if os.path.exists("rows_board.f32") and os.path.exists("rows_board2.f32"): cmp(ld("rows_board.f32"), ld("rows_board2.f32"), "board lần 1 vs lần 2 (xác định?)")

## D · TXT — chữ sinh của ref (INT8 mô phỏng) cho một số clip  (đã xong, có thể bỏ qua)
`--hp` = float64 cho norm/softmax/GELU/RoPE/SiLU, tức một mẫu nhiễu ulp khác. Sửa `CLIPS` rồi chạy (mỗi lệnh 1–2 phút với 40–80 token).

In [ ]:
# [D1] Chữ ref cho các clip trong CLIPS, mỗi clip hai mẫu nhiễu (float32 / --hp)
CLIPS = ["03.wav", "04.wav"]      # cần upload các file wav này
MAXNEW = 80
for w in CLIPS:
    for hp in ("", "--hp"):
        print(f"--- {w} | {hp or 'float32'}")
        !python3 ref_runtime.py run --pkg {PKG} --wav {w} --max-new {MAXNEW} {hp} 2>&1 | tail -3

## E · SCORE — chấm điểm cả bộ trên board so với câu gốc FP32  (đã xong, có thể bỏ qua)
Kết quả (14 clip, 4.69 token/s, 3 clip bị gắn cờ) đã ghi trong Project. Chạy lại khi có `results.tsv` mới từ `run_wav_set.sh`.

In [ ]:
# [E1] Upload score_wav_set.py, answers.txt, results.tsv rồi chấm điểm
from google.colab import files; import os, shutil
up = files.upload()                                   # score_wav_set.py, answers.txt, results.tsv
os.makedirs("wavset_out", exist_ok=True)
if "results.tsv" in up: shutil.move("results.tsv", "wavset_out/results.tsv")
!python3 score_wav_set.py --answers answers.txt --tsv wavset_out/results.tsv --out report.md --csv scores.csv
from IPython.display import Markdown; Markdown(open("report.md", encoding="utf-8").read())

## F · SWAP — FP32 ↔ INT8: lỗi "mất video game" (clip 01) nằm ở encoder hay LLM?
Câu gốc của `01.wav` (FP32 thuần) nhắc "video game"; cả ba mẫu INT8 (board, ref float32, ref `--hp`) đều bỏ mất. Hai thí nghiệm hoán đổi:

| | Encoder FP32 | Encoder INT8 |
|---|---|---|
| **LLM FP32** | đúng (câu gốc) | **E3**: ô F5 |
| **LLM INT8** | **E2**: chạy trên board (ô F6) | sai (board, ref) |

Cần upload: `01.wav` và `rows_board.f32` (từ board, lần chạy `--wav` trên `01.wav` có `--dump-rows rows_board.f32`).
Mô hình FP32 được nạp từ `MANIFEST.json` của gói (id + revision) — tải ~4–5 GB, 1–3 phút; dùng GPU nếu có.

In [ ]:
# [F1] Nạp mô hình FP32 gốc (id + revision lấy từ MANIFEST của gói), tokenizer và feature extractor
import json, sys, torch, numpy as np
sys.path.insert(0, ".")
import ref_runtime as R
from transformers import AutoTokenizer, WhisperFeatureExtractor
man = json.load(open(f"{PKG}/MANIFEST.json")); MID, REV, dec = man["model"]["id"], man["model"]["revision"], man["decode"]
print("model:", MID, "| revision:", REV, "| transformers khi tạo gói:", man["model"].get("transformers"), "| bản hiện tại:", __import__("transformers").__version__)
print("giải mã:", {k: dec[k] for k in ("greedy", "repetition_penalty", "eos_id", "audio_token_id", "max_new")})
dev = "cuda" if torch.cuda.is_available() else "cpu"
try:
    from transformers import Qwen2AudioForConditionalGeneration as MC
except ImportError:
    from transformers import AutoModelForCausalLM as MC
model = MC.from_pretrained(MID, revision=REV, torch_dtype=torch.float32, trust_remote_code=True).to(dev).eval()
tok = AutoTokenizer.from_pretrained(f"{PKG}/hf_processor"); fe = WhisperFeatureExtractor.from_pretrained(f"{PKG}/hf_processor")
print(type(model).__name__, "| trên", dev, "| các khối con:", [n for n, _ in model.named_children()])

model: tranquangchung/qwen2-audio-dialogue | revision: 01061daa248750e56b43801a3c319fb4e6443704 | transformers khi tạo gói: 4.55.0 | bản hiện tại: 5.22.1
giải mã: {'greedy': True, 'repetition_penalty': 1.1, 'eos_id': 151645, 'audio_token_id': 151646, 'max_new': 200}


config.json:   0%|          | 0.00/1.99k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/69.7k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/780 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/121 [00:00<?, ?B/s]

Qwen2AudioForConditionalGeneration | trên cpu | các khối con: ['model', 'lm_head']


In [13]:
# [F2] Hàm dựng đầu vào từ WAV (cùng cách đọc/resample và cùng prompt như ref/board) + hook lấy/thay hàng audio ở đầu ra projector
cap = {}
def _hook(m, inp, out):
    cap["rows"] = out.detach().float().cpu()
    inj = cap.get("inject")
    if inj is not None:                                   # E3: thay các hàng audio hợp lệ bằng hàng encoder INT8
        flat = out.clone().reshape(-1, out.shape[-1]); flat[:inj.shape[0]] = torch.from_numpy(inj).to(flat.dtype).to(flat.device)
        return flat.reshape(out.shape)
try: hk.remove()
except NameError: pass
_proj = [m for n, m in model.named_modules() if n.split(".")[-1] == "multi_modal_projector"]     # transformers 4.x: model.multi_modal_projector, 5.x: model.model.multi_modal_projector
assert len(_proj) == 1, ("không tìm thấy đúng 1 multi_modal_projector", [n for n, _ in model.named_modules() if "proj" in n][:10])
hk = _proj[0].register_forward_hook(_hook)

def build_inputs(wav, wave=None):
    if wave is None: wave = R.read_wav(wav)
    f = fe(wave, sampling_rate=16000, return_tensors="pt", return_attention_mask=True)       # đệm tới 30 s như lúc sinh golden
    L = int(f["attention_mask"].sum()); n = ((L - 1) // 2 + 1 - 2) // 2 + 1                  # số khung mel hợp lệ -> số hàng audio
    ids = torch.tensor([tok(dec["prompt"].replace("<|AUDIO|>", "<|AUDIO|>" * n), add_special_tokens=False)["input_ids"]])
    assert int((ids == dec["audio_token_id"]).sum()) == n
    inp = dict(input_ids=ids, attention_mask=torch.ones_like(ids), input_features=f["input_features"], feature_attention_mask=f["attention_mask"])
    return {k: v.to(dev) for k, v in inp.items()}, L, n, wave

def gen(inp, max_new=60):
    with torch.no_grad():
        out = model.generate(**inp, max_new_tokens=max_new, do_sample=False, repetition_penalty=dec["repetition_penalty"], eos_token_id=dec["eos_id"], pad_token_id=dec["eos_id"])
    return tok.decode(out[0, inp["input_ids"].shape[1]:], skip_special_tokens=True).strip()

In [ ]:
# [F3] FP32 trên 01.wav: lấy hàng audio FP32 (E1/E2) và kiểm tra pipeline có tái tạo câu gốc FP32 không
WAV = "01.wav"
inp, L, n, wave = build_inputs(WAV)
print("mel hợp lệ:", L, "-> hàng audio:", n, "(board: mel 688 -> 172)")
cap.pop("inject", None)
with torch.no_grad(): model(**inp)
rows = cap["rows"].reshape(-1, cap["rows"].shape[-1])[:n]
print("hàng audio FP32:", tuple(rows.shape))
rows.numpy().astype("<f4").tofile("rows_fp32_01.f32")
print("\nFP32 (câu sinh lại):", gen(inp, 60))
print("\n(so với câu gốc của 01.wav: phải nhắc 'video game'; nếu khác hẳn thì prompt/tham số giải mã ở đây chưa giống lúc sinh câu gốc)")

mel hợp lệ: 688 -> hàng audio: 172 (board: mel 688 -> 172)
hàng audio FP32: (172, 896)

FP32 (câu sinh lại): That's an interesting question! Personally, I think the universe would be a very small one, like a tiny speck on a cosmic scale. Given how vast the universe is, it seems like there would be plenty of room for endless possibilities. However, if I had to choose, I might go

(so với câu gốc của 01.wav: phải nhắc 'video game'; nếu khác hẳn thì prompt/tham số giải mã ở đây chưa giống lúc sinh câu gốc)


In [ ]:
# [F4] E1: encoder INT8 của board lệch FP32 bao nhiêu (so với nhiễu 0.13 giữa các mẫu INT8)
import os
f32 = np.fromfile("rows_fp32_01.f32", np.float32).reshape(-1, rows.shape[1]); b = np.fromfile("rows_board.f32", np.float32).reshape(-1, rows.shape[1])
assert f32.shape == b.shape, (f32.shape, b.shape)
d = b - f32
print(f"INT8 board vs FP32: rms tương đối {np.sqrt((d**2).mean()/(f32**2).mean()):.3f} | max|diff| {abs(d).max():.3f} | cos hàng min {min((x@y)/np.linalg.norm(x)/np.linalg.norm(y) for x, y in zip(b, f32)):.3f}")
for nm, fn in [("ref float32", "rows_ref.f32"), ("ref float64", "rows_ref_hp.f32")]:
    if os.path.exists(fn):
        r = np.fromfile(fn, np.float32).reshape(-1, rows.shape[1]); d = r - f32
        print(f"INT8 {nm:11s} vs FP32: rms tương đối {np.sqrt((d**2).mean()/(f32**2).mean()):.3f}")

INT8 board vs FP32: rms tương đối 0.170 | max|diff| 1.985 | cos hàng min 0.660


In [ ]:
# [F5] E3: LLM FP32 + hàng encoder INT8 của board (rows_board.f32)
cap["inject"] = np.fromfile("rows_board.f32", np.float32).reshape(-1, rows.shape[1])
print("E3 (LLM FP32 + encoder INT8):", gen(inp, 60))
cap.pop("inject", None)

E3 (LLM FP32 + encoder INT8): That's an interesting question! Personally, I think the universe would be a very peaceful one. It would have minimal conflict between different worlds, which could make life much more harmonious. However, if you're looking for something with more excitement or adventure, perhaps a world where humanity is constantly on the


array([[ 0.6820754 ,  0.14592211,  0.8964406 , ...,  1.5076996 ,
         0.19081353, -1.5959704 ],
       [-0.43078446, -0.29184422,  0.3897568 , ...,  0.58837056,
         0.53427786,  1.856537  ],
       [ 0.14359482,  0.5472079 ,  0.23385407, ..., -0.11031948,
         0.648766  , -0.09771247],
       ...,
       [-1.3641508 ,  1.1308963 , -1.1692704 , ..., -0.77223635,
         0.7250914 ,  0.42342073],
       [-0.7897715 ,  1.7510653 , -0.7795136 , ..., -0.99287534,
         0.7250914 ,  0.09771247],
       [ 0.21539223,  1.313299  , -0.3897568 , ..., -0.44127792,
         0.30530164,  0.6514165 ]], shape=(172, 896), dtype=float32)

In [ ]:
# [F6] Tải hàng FP32 về để chạy E2 trên board (LLM INT8 + encoder FP32)
from google.colab import files; files.download("rows_fp32_01.f32")

**E2 trên board KR260-04** (sau khi chép `rows_fp32_01.f32` vào `~/dduong/kr260/tools`, dùng `llm_runtime.cpp` v8b):
```
sudo env NPU_PL_OK=1 ./llm_runtime --pkg ../pkg_v1 --audio rows_fp32_01.f32 --template in_clip00/ids.i32 --backend npu --max-new 60 --threads 4 --text-out e2_01.txt 2>&1 | grep -E "\[audio\]|DECODE"; cat e2_01.txt
```
Cách đọc: E2 nhắc lại "video game" → lỗi nằm ở encoder INT8 (hướng sửa: tăng độ chính xác lượng tử cho encoder). E2 vẫn mất mà E3 đúng → lỗi nằm ở LLM INT8. Cả hai mất → cộng dồn. Mỗi ô là một mẫu duy nhất và mô hình rất nhạy nhiễu, nên nếu không rõ thì lặp lại với clip 03.

## G · FRONT — front-end âm thanh (đọc WAV → resample → log-mel) có khớp với pipeline gốc không?
**Phát hiện từ F3:** mô hình **FP32** chạy trên đặc trưng do `ref_runtime.read_wav` tạo ra cũng KHÔNG nhắc "video game" (nó sinh gần giống hệt ref INT8: "…very small one, like a tiny speck on a cosmic scale…"), còn câu gốc FP32 thì có. Vậy INT8 chưa phải thủ phạm; nghi vấn chuyển sang **đầu vào**: `read_wav` (cũng là code của board) đổi tần số lấy mẫu bằng nội suy tuyến tính, không lọc chống chồng phổ.
Bằng chứng đã có: hàng audio từ `01.wav` lệch hàng audio của golden clip 0 (rms 0.63) dù hai bên cùng dài 688 khung mel (rất có thể là cùng một file).
Các ô G1 → G3 kiểm tra điều đó (cần F1, F2 đã chạy; `01.wav` đã upload).

In [14]:
# [G1] 01.wav -> mel: cách đọc/resample của ref_runtime so với golden clip 0 và so với resampler chuẩn
import soundfile as sf, scipy.signal as ss, math
meta = json.load(open(f"{PKG}/golden/clip_00/meta.json")); print("golden clip 0: wav =", meta.get("wav"), "| câu hỏi:", meta.get("question"))
w0, sr0 = sf.read("01.wav", dtype="float32"); mono = w0.mean(1) if w0.ndim > 1 else w0
print(f"01.wav: tần số {sr0} Hz | {'mono' if w0.ndim == 1 else str(w0.shape[1]) + ' kênh'} | {len(w0)/sr0:.2f} s | biên độ max {abs(w0).max():.3f}")
g = np.fromfile(f"{PKG}/golden/clip_00/input_features.f32", "<f4").reshape(128, 3000)
L0 = meta.get("valid", {}).get("mel", 688)
feats = lambda w: fe(w, sampling_rate=16000, return_tensors="np", return_attention_mask=True)["input_features"][0]
cands = {"ref_runtime.read_wav (nội suy tuyến tính)": R.read_wav("01.wav")}
if sr0 != 16000:
    g_ = math.gcd(16000, sr0); cands["scipy.resample_poly (chuẩn)"] = ss.resample_poly(mono, 16000 // g_, sr0 // g_).astype(np.float32)
try:
    import librosa; cands["librosa.load(sr=16000)"] = librosa.load("01.wav", sr=16000)[0].astype(np.float32)
except Exception as e: print("không dùng được librosa:", e)
for k, w in cands.items():
    f = feats(w); n = min(L0, f.shape[1], g.shape[1]); d = f[:, :n] - g[:, :n]
    print(f"{k:44s} | {len(w)} mẫu | mel vs golden: max|d| {abs(d).max():.3e}  rms {np.sqrt((d**2).mean()):.3e}")

golden clip 0: wav = 01.wav | câu hỏi: If you had to live in a video game universe for a year, which one would you choose?
01.wav: tần số 24000 Hz | mono | 6.88 s | biên độ max 0.824
ref_runtime.read_wav (nội suy tuyến tính)    | 110080 mẫu | mel vs golden: max|d| 6.227e-01  rms 3.047e-02
scipy.resample_poly (chuẩn)                  | 110080 mẫu | mel vs golden: max|d| 4.484e-01  rms 8.371e-03
librosa.load(sr=16000)                       | 110080 mẫu | mel vs golden: max|d| 5.233e-05  rms 5.781e-07


In [15]:
# [G2] Kiểm tra pipeline FP32: đưa ĐẶC TRƯNG GOLDEN vào mô hình FP32 -> có tái tạo token FP32 của golden không? (~2-4 phút trên CPU)
gd = f"{PKG}/golden/clip_00"
ids0 = np.fromfile(gd + "/input_ids.i32", "<i4"); mask0 = np.fromfile(gd + "/feature_attention_mask.i32", "<i4"); gf = np.fromfile(gd + "/fp32_gen_ids.i32", "<i4")
inp0 = dict(input_ids=torch.tensor(ids0)[None].long(), attention_mask=torch.ones(1, len(ids0), dtype=torch.long), input_features=torch.tensor(g)[None], feature_attention_mask=torch.tensor(mask0)[None].long())
inp0 = {k: v.to(dev) for k, v in inp0.items()}
NG = min(len(gf), 40)
with torch.no_grad():
    out = model.generate(**inp0, max_new_tokens=NG, do_sample=False, repetition_penalty=dec["repetition_penalty"], eos_token_id=dec["eos_id"], pad_token_id=dec["eos_id"])
new = out[0, inp0["input_ids"].shape[1]:].cpu().numpy()
m = min(len(new), NG); k = int((new[:m] == gf[:m]).cumprod().sum())
print(f"trùng token FP32 của golden: {k}/{m} token đầu liên tiếp")
print("FP32 trên đặc trưng golden:", tok.decode(new, skip_special_tokens=True).strip())

trùng token FP32 của golden: 40/40 token đầu liên tiếp
FP32 trên đặc trưng golden: That's an interesting question! Personally, I think the universe would be incredibly peaceful if it were a good old-fashioned video game world. There wouldn't be any conflict or violence between players. On the


In [16]:
# [G3] FP32 trên 01.wav với resampler chuẩn (librosa hoặc scipy) thay cho nội suy tuyến tính
best = cands.get("librosa.load(sr=16000)", cands.get("scipy.resample_poly (chuẩn)", cands["ref_runtime.read_wav (nội suy tuyến tính)"]))
inp1, L1, n1, _ = build_inputs("01.wav", wave=best); print("mel hợp lệ:", L1, "-> hàng audio:", n1)
print("FP32, front-end chuẩn:", gen(inp1, 60))

mel hợp lệ: 688 -> hàng audio: 172
FP32, front-end chuẩn: That's an interesting question! Personally, I think the universe would be incredibly peaceful if it were a good old-fashioned video game world. There wouldn't be any conflict or violence between players. On the other hand, some people might find that a more realistic or fair scenario could have more consequences. It
